# 06 — Neuronpedia Feature Fingerprint Analysis

**No GPU required.** Run **top to bottom** once (first run fetches Neuronpedia labels, ~10–15 min).

Reads `top50_features` from the statistics JSONs, fetches Neuronpedia descriptions for every unique
`(layer, feat_idx)`, classifies them into six semantic categories, then computes **fingerprints**
(category proportions) per phase and produces the primary visualisations for §5B of the thesis.

| Phase | Stats file |
|-------|------------|
| `base` | `stats_base_v2.json` (159 prompts) |
| `lora_l1` | `stats_lora_l1.json` (300 prompts) |
| `lora_l2` | `stats_lora_l2.json` (300 prompts) |
| `lora_l3_tri_num` | `stats_lora_l3_tri_num.json` (284 prompts) |

**Sections:**  
0. Paths · 1. Load stats · 2. Collect features · 3. Batch-fetch Neuronpedia ·
4. Inspect raw labels · 5. Category distribution · 6. Fingerprints (overall + True/False split) ·
7. Vis A — fingerprint bar chart · 8. Vis B — category × position heatmaps ·
9. Vis C — top-10 features per category · 10. Manual overrides · 11. Export outputs

**Hypotheses tested (§5B):**  
H5 — base fingerprint dominated by `format-template`  
H6 — after LoRA on triangle data, `geometry` proportion increases  
H7 — after LoRA on broad math, `math-general` increases (not `geometry`)  
H8 — `boolean-logic` increases after both fine-tuning runs


## 0 — Environment setup

In [ ]:
import os
import sys
from pathlib import Path

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root : {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo. Set CT_REPO_DIR env var.")

MY_WORK     = _my_work if _root else Path(".").resolve()
STATS_DIR   = MY_WORK / "results" / "statistics"
FIGURES_DIR = MY_WORK / "results" / "figures" / "neuronpedia"
CACHE_PATH  = MY_WORK / "cache" / "neuronpedia_cache.json"
OVERRIDES_PATH = MY_WORK / "cache" / "category_overrides.json"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"MY_WORK     : {MY_WORK}")
print(f"STATS_DIR   : {STATS_DIR}")
print(f"FIGURES_DIR : {FIGURES_DIR}")
print(f"CACHE_PATH  : {CACHE_PATH}")


## 1 — Load statistics files

Registers all four phases. If `stats_base_v3.json` (from notebook 02c) is synced back later,
add it here as phase `base_v3` — no other changes required.

In [ ]:
import json
from utils.graph_statistics import load_statistics

# ── Phase registry ─────────────────────────────────────────────────────────────
PHASE_FILES = {
    "base":            STATS_DIR / "stats_base_v2.json",
    "lora_l1":         STATS_DIR / "stats_lora_l1.json",
    "lora_l2":         STATS_DIR / "stats_lora_l2.json",
    "lora_l3_tri_num": STATS_DIR / "stats_lora_l3_tri_num.json",
    # Uncomment if/when 02c results are synced back:
    # "base_v3": STATS_DIR / "stats_base_v3.json",
}

phase_stats: dict[str, list[dict]] = {}
for phase, path in PHASE_FILES.items():
    if not path.exists():
        print(f"WARNING: {path} not found — skipping phase '{phase}'")
        continue
    entries = load_statistics(path)
    succeeded = [e for e in entries if e.get("attribution_succeeded")]
    phase_stats[phase] = entries
    print(f"{phase:20s}: {len(entries):4d} total, {len(succeeded):4d} succeeded")

print(f"\nPhases loaded: {list(phase_stats.keys())}")


## 2 — Collect unique (layer, feat_idx) pairs

Extracts every `(layer, feat_idx)` from `top50_features` across all phases.
Position (`pos`) is **not** part of the Neuronpedia cache key — position is per-prompt context,
not a property of the feature itself.

In [ ]:
# Per-phase unique (layer, feat_idx)
phase_features: dict[str, set[tuple[int, int]]] = {}
all_features: set[tuple[int, int]] = set()

for phase, entries in phase_stats.items():
    feats: set[tuple[int, int]] = set()
    for e in entries:
        if not e.get("attribution_succeeded"):
            continue
        for triple in (e.get("top50_features") or []):
            layer, _pos, feat_idx = triple[0], triple[1], triple[2]
            feats.add((layer, feat_idx))
    phase_features[phase] = feats
    all_features |= feats
    print(f"{phase:20s}: {len(feats):4d} unique (layer, feat_idx) pairs")

print(f"\nAll phases combined: {len(all_features)} unique pairs to look up")


## 3 — Batch-fetch Neuronpedia labels

Uses `utils/neuronpedia.py` → `fetch_and_cache_batch`.  
Rate-limited to ≥ 0.5 s between requests; cache written every 10 new fetches.  
**First run**: ~10–15 min. **Subsequent runs**: instant (all served from cache).

Neuronpedia endpoint:  
`GET https://www.neuronpedia.org/api/feature/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feat_idx}`

In [ ]:
import json
from utils.neuronpedia import load_cache, save_cache, fetch_and_cache_batch

# Load manual overrides (empty dict if file is missing or empty)
override_map: dict = {}
if OVERRIDES_PATH.exists():
    with open(OVERRIDES_PATH) as f:
        override_map = json.load(f) or {}
print(f"Overrides loaded: {len(override_map)} entries")

cache = load_cache(CACHE_PATH)
already_cached = len(cache)
to_fetch = [pair for pair in all_features if f"{pair[0]}_{pair[1]}" not in cache]
print(f"Cache: {already_cached} existing entries, {len(to_fetch)} new fetches needed")

if to_fetch:
    print("Fetching from Neuronpedia (this will take a while on first run)...")
    cache = fetch_and_cache_batch(
        list(all_features), cache, CACHE_PATH, override_map=override_map
    )
    print(f"Fetch complete. Cache now has {len(cache)} entries.")
else:
    print("All features already cached — no network requests needed.")


## 4 — Inspect raw labels

Sanity-check: print a random sample of the fetched Neuronpedia labels per phase.
Use this section to spot misclassifications before proceeding to Section 10 (overrides).

In [ ]:
import random
import pandas as pd

SAMPLE_N = 15  # features to sample per phase
random.seed(42)

for phase, feats in phase_features.items():
    sample = random.sample(sorted(feats), min(SAMPLE_N, len(feats)))
    rows = []
    for layer, feat_idx in sorted(sample):
        key = f"{layer}_{feat_idx}"
        entry = cache.get(key, {})
        rows.append({
            "layer": layer,
            "feat_idx": feat_idx,
            "label": entry.get("label", "(missing)"),
            "category": entry.get("category", "?"),
            "top_tokens": ", ".join((entry.get("top_tokens") or [])[:5]),
        })
    df = pd.DataFrame(rows)
    print(f"\n=== {phase} (sample of {len(rows)}) ===")
    pd.set_option("display.max_colwidth", 80)
    display(df)


## 5 — Category distribution

Show the raw count of each category across all features in each phase's `top50_features`.

In [ ]:
from collections import defaultdict, Counter
from utils.feature_categorizer import categorize_label, apply_category_overrides

CATEGORIES = ["geometry", "boolean-logic", "language-comparative",
              "math-general", "format-template", "other"]

def _get_category(layer: int, feat_idx: int) -> str:
    """Look up category from cache, applying overrides."""
    key = f"{layer}_{feat_idx}"
    entry = cache.get(key, {})
    cat = entry.get("category") or categorize_label(entry.get("label", ""))
    return apply_category_overrides(layer, feat_idx, cat, override_map)

# Appearance counts: each (layer, pos, feat_idx) occurrence in top50_features counts once
phase_cat_counts: dict[str, Counter] = {}
for phase, entries in phase_stats.items():
    counts: Counter = Counter()
    for e in entries:
        if not e.get("attribution_succeeded"):
            continue
        for triple in (e.get("top50_features") or []):
            layer, _pos, feat_idx = triple[0], triple[1], triple[2]
            counts[_get_category(layer, feat_idx)] += 1
    phase_cat_counts[phase] = counts

# Summary table
rows = []
for phase, counts in phase_cat_counts.items():
    total = sum(counts.values())
    row = {"phase": phase, "total_appearances": total}
    for cat in CATEGORIES:
        row[cat] = counts.get(cat, 0)
        row[f"{cat}_%"] = round(100 * counts.get(cat, 0) / total, 1) if total else 0
    rows.append(row)

df_counts = pd.DataFrame(rows).set_index("phase")
print("Category appearance counts across top50_features per phase:")
display(df_counts[["total_appearances"] + CATEGORIES])
print("\nPercentages:")
display(df_counts[[f"{c}_%" for c in CATEGORIES]])


## 6 — Fingerprints (overall + True/False split)

Compute one fingerprint per phase (category proportions, summing to 1).  
Also split by `label` (True vs False prompts) to test H5/H8.

In [ ]:
from utils.feature_categorizer import compute_fingerprint

# ── Overall fingerprints ───────────────────────────────────────────────────────
fingerprints: dict[str, dict] = {}
for phase, entries in phase_stats.items():
    fingerprints[phase] = compute_fingerprint(entries, cache, override_map=override_map)

df_fp = pd.DataFrame(fingerprints).T[CATEGORIES]
df_fp = df_fp.round(4)
print("Overall fingerprints (proportion per category):")
display(df_fp)

# ── True/False split ───────────────────────────────────────────────────────────
fp_split: dict[str, dict] = {}  # keys like "base/True", "base/False"

for phase, entries in phase_stats.items():
    for label_val in [True, False]:
        subset = [
            e for e in entries
            if e.get("attribution_succeeded")
            and bool(e.get("label")) == label_val
        ]
        if subset:
            fp_split[f"{phase}/{'True' if label_val else 'False'}"] = (
                compute_fingerprint(subset, cache, override_map=override_map)
            )

df_split = pd.DataFrame(fp_split).T[CATEGORIES].round(4)
print("\nFingerprints split by label (True/False):")
display(df_split)


## 7 — Visualisation A: Fingerprint bar chart

Grouped bar chart: categories on x-axis, phases as bar groups.  
Saved to `results/figures/neuronpedia/fingerprint_bar.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PHASE_LABELS = {
    "base":            "Base",
    "lora_l1":         "LoRA L1",
    "lora_l2":         "LoRA L2",
    "lora_l3_tri_num": "LoRA L3 (tri+num)",
}
CAT_COLORS = {
    "geometry":            "#4e79a7",
    "boolean-logic":       "#f28e2b",
    "language-comparative":"#e15759",
    "math-general":        "#76b7b2",
    "format-template":     "#59a14f",
    "other":               "#b0b0b0",
}

phases = list(fingerprints.keys())
x = np.arange(len(CATEGORIES))
n_phases = len(phases)
width = 0.18
offsets = np.linspace(-(n_phases - 1) / 2, (n_phases - 1) / 2, n_phases) * width

fig, ax = plt.subplots(figsize=(12, 5))
for i, phase in enumerate(phases):
    vals = [fingerprints[phase].get(cat, 0) for cat in CATEGORIES]
    bars = ax.bar(
        x + offsets[i], vals, width,
        label=PHASE_LABELS.get(phase, phase),
        color=[CAT_COLORS[c] for c in CATEGORIES],
        alpha=0.85,
        edgecolor="white",
    )

ax.set_xticks(x)
ax.set_xticklabels(CATEGORIES, rotation=25, ha="right", fontsize=10)
ax.set_ylabel("Proportion of top-50 feature appearances")
ax.set_title("Feature Category Fingerprints — Base vs LoRA Phases")
ax.set_ylim(0, 1)
ax.legend(title="Phase", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

fig.tight_layout()
out_path = FIGURES_DIR / "fingerprint_bar.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()


### 7b — Stacked bar chart (alternative view)

Each phase is a single bar; categories stacked to show the full composition at a glance.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
phase_display = [PHASE_LABELS.get(p, p) for p in phases]
bottoms = np.zeros(len(phases))

for cat in CATEGORIES:
    vals = np.array([fingerprints[p].get(cat, 0) for p in phases])
    ax.bar(phase_display, vals, bottom=bottoms, label=cat,
           color=CAT_COLORS[cat], edgecolor="white", alpha=0.9)
    bottoms += vals

ax.set_ylabel("Proportion of top-50 feature appearances")
ax.set_title("Feature Category Fingerprints (stacked) — Base vs LoRA")
ax.set_ylim(0, 1)
ax.legend(title="Category", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

fig.tight_layout()
out_path = FIGURES_DIR / "fingerprint_stacked.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()


## 8 — Visualisation B: Category × token-position heatmaps

For each phase: a heatmap with categories on the y-axis and token positions (0–29) on the x-axis.  
Cells show normalised count of feature appearances at each (category, position).

Key question: does `format-template` shift away from pos=21 (Answer: slot) toward content
token positions (pos 5–18) after fine-tuning?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from utils.feature_categorizer import compute_position_heatmap

MAX_POS = 35  # show positions 0–34; extend if prompts are longer

for phase, entries in phase_stats.items():
    pos_data = compute_position_heatmap(entries, cache, override_map=override_map)

    # Build matrix: rows = categories, cols = positions
    matrix = np.zeros((len(CATEGORIES), MAX_POS))
    for r, cat in enumerate(CATEGORIES):
        for pos, count in (pos_data.get(cat) or {}).items():
            if pos < MAX_POS:
                matrix[r, pos] = count

    # Normalise each row to 0-1 (proportion across positions)
    row_sums = matrix.sum(axis=1, keepdims=True)
    matrix_norm = np.where(row_sums > 0, matrix / row_sums, 0)

    fig, ax = plt.subplots(figsize=(14, 3.5))
    im = ax.imshow(matrix_norm, aspect="auto", cmap="Blues", vmin=0, vmax=matrix_norm.max() or 1)
    ax.set_yticks(range(len(CATEGORIES)))
    ax.set_yticklabels(CATEGORIES, fontsize=9)
    ax.set_xticks(range(0, MAX_POS, 2))
    ax.set_xticklabels(range(0, MAX_POS, 2), fontsize=8)
    ax.set_xlabel("Token position")
    ax.set_title(f"Category × Token Position — {PHASE_LABELS.get(phase, phase)}")
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="Normalised count")
    ax.axvline(x=21 - 0.5, color="red", linewidth=1.2, linestyle="--", alpha=0.7,
               label="pos=21 (Answer: slot)")
    ax.legend(loc="upper right", fontsize=8)

    fig.tight_layout()
    out_path = FIGURES_DIR / f"position_heatmap_{phase}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()


## 9 — Visualisation C: Top-10 features per category

For each phase, list the 10 most frequently appearing `(layer, feat_idx)` pairs in each category,
with their Neuronpedia labels and clickable dashboard URLs.  
Provides qualitative grounding for the fingerprint numbers.

In [ ]:
from IPython.display import Markdown, display as ipy_display
from utils.feature_categorizer import get_top_features_by_category

NP_URL_TEMPLATE = (
    "https://www.neuronpedia.org/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feat_idx}"
)
TOP_N = 10

all_top_rows = []  # for CSV export in Section 11

for phase, entries in phase_stats.items():
    ipy_display(Markdown(f"### Phase: {PHASE_LABELS.get(phase, phase)}"))
    for cat in CATEGORIES:
        top = get_top_features_by_category(
            entries, cache, cat, n=TOP_N, override_map=override_map
        )
        if not top:
            continue
        rows_md = ["| Layer | feat_idx | Count | Label | URL |"]
        rows_md.append("|-------|----------|-------|-------|-----|")
        for r in top:
            url = NP_URL_TEMPLATE.format(layer=r["layer"], feat_idx=r["feat_idx"])
            label_trunc = (r["label"] or "")[:80]
            rows_md.append(
                f"| {r['layer']} | {r['feat_idx']} | {r['count']} "
                f"| {label_trunc} | [↗]({url}) |"
            )
            all_top_rows.append({
                "phase": phase, "category": cat,
                "layer": r["layer"], "feat_idx": r["feat_idx"],
                "count": r["count"], "label": r["label"],
                "url": url,
            })
        ipy_display(Markdown(f"**{cat}**\n" + "\n".join(rows_md) + "\n"))


## 10 — Manual overrides

After inspecting Section 4 / Section 9, edit `my_work/cache/category_overrides.json` to correct
any misclassified features. Format:

```json
{
  "24_4640": "geometry",
  "18_6735": "boolean-logic"
}
```

Then **re-run from Section 5** (no new Neuronpedia fetches needed — overrides are applied at
classification time). The cache itself is **not** rewritten; overrides are a separate layer.

The cell below reloads the override file so you can iterate without restarting the kernel.

In [ ]:
# Reload overrides after editing category_overrides.json
with open(OVERRIDES_PATH) as f:
    override_map = json.load(f) or {}
print(f"Reloaded {len(override_map)} override entries.")
print("→ Re-run cells from Section 5 to apply.")


## 11 — Export outputs

Saves CSV tables for use in the thesis and supplementary material.

In [ ]:
import pandas as pd

# ── Fingerprint table ─────────────────────────────────────────────────────────
fp_out = FIGURES_DIR / "fingerprint_by_phase.csv"
df_fp.to_csv(fp_out)
print(f"Saved: {fp_out}")

# ── True/False split table ────────────────────────────────────────────────────
split_out = FIGURES_DIR / "fingerprint_true_false_split.csv"
df_split.to_csv(split_out)
print(f"Saved: {split_out}")

# ── Category counts table ─────────────────────────────────────────────────────
counts_out = FIGURES_DIR / "category_counts_by_phase.csv"
df_counts.to_csv(counts_out)
print(f"Saved: {counts_out}")

# ── Top-10 features per category ─────────────────────────────────────────────
top10_out = FIGURES_DIR / "top10_features_per_category.csv"
pd.DataFrame(all_top_rows).to_csv(top10_out, index=False)
print(f"Saved: {top10_out}")

print("\n=== All outputs saved ===")
print(f"Directory: {FIGURES_DIR}")
for f in sorted(FIGURES_DIR.iterdir()):
    print(f"  {f.name}")
